In [ ]:
!pip install transformers torch peft sentencepiece -q

In [ ]:
import json, os, torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

# Charger le modèle fine-tuné de ta collègue
MODEL_NAME   = "google/pegasus-xsum"
MODEL_DIR    = "model_finetuned"

print("🔄 Chargement du modèle PEGASUS + LoRA...")
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model      = PeftModel.from_pretrained(base_model, MODEL_DIR)
tokenizer  = AutoTokenizer.from_pretrained(MODEL_DIR)

device = "cuda" if torch.cuda.is_available() else "cpu"
model  = model.to(device)
model.eval()

print(f"✅ Modèle chargé sur {device.upper()}")

In [ ]:
import json, os

# Détecter le bon chemin
if os.path.exists("donnees_propres.json"):
    chemin = "donnees_propres.json"
elif os.path.exists("notebooks/donnees_propres.json"):
    chemin = "notebooks/donnees_propres.json"
else:
    raise FileNotFoundError("❌ donnees_propres.json introuvable !")

with open(chemin, "r", encoding="utf-8") as f:
    donnees = json.load(f)

print(f"✅ {len(donnees)} livres chargés")
print(f"   Exemple 1 : {donnees[0]['nb_chunks']} chunks")

In [ ]:
def resumer_chunk(chunk, max_input=512, max_output=128):
    inputs = tokenizer(
        chunk,
        max_length=max_input,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_output,
            num_beams=2,
            early_stopping=True,
            no_repeat_ngram_size=3,
            forced_bos_token_id=None   # ← supprime le conflit max_length
        )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("✅ Fonction resumer_chunk définie !")

In [ ]:
def pipeline_mapreduce(exemple):
    """
    MAP    : résumer chaque chunk individuellement
    REDUCE : résumer tous les résumés en un résumé final
    """
    chunks = exemple["chunks"]
    print(f"   📚 {len(chunks)} chunks à traiter...")
    
    # ── MAP : résumer chaque chunk ──
    resumes_intermediaires = []
    for i, chunk in enumerate(chunks):
        print(f"   🔄 Chunk {i+1}/{len(chunks)}...", end=" ")
        resume = resumer_chunk(chunk)
        resumes_intermediaires.append(resume)
        print(f"✅ ({len(resume.split())} mots)")
    
    # ── REDUCE : combiner les résumés intermédiaires ──
    texte_combine = " ".join(resumes_intermediaires)
    print(f"   🔄 Reduce : résumé final...")
    resume_final = resumer_chunk(texte_combine, max_input=512, max_output=150)
    
    return {
        "resumes_chunks": resumes_intermediaires,
        "resume_final":   resume_final
    }

print("✅ Fonction pipeline_mapreduce définie !")

In [ ]:
print("=" * 55)
print("TEST PIPELINE MAP-REDUCE sur 3 livres")
print("=" * 55)

resultats = []

for i in range(3):
    print(f"\n📖 Livre {i+1} :")
    res = pipeline_mapreduce(donnees[i])
    
    print(f"\n   ✅ Résumé final ({len(res['resume_final'].split())} mots) :")
    print(f"   {res['resume_final']}")
    print(f"\n   📌 Référence :")
    print(f"   {donnees[i]['resume'][:200]}...")
    
    resultats.append({
        "index":        i,
        "resume_final": res["resume_final"],
        "reference":    donnees[i]["resume"]
    })

print("\n✅ Pipeline Map-Reduce terminé !")

In [ ]:
import json

# Sauvegarder les résultats
with open("resultats_mapreduce.json", "w", encoding="utf-8") as f:
    json.dump(resultats, f, indent=2, ensure_ascii=False)

print("✅ resultats_mapreduce.json sauvegardé !")
print(f"   {len(resultats)} livres résumés")